<a href="https://colab.research.google.com/github/GutsAtGit/anomaly_detection_ema_gnn/blob/main/c10_m1_t0p5_full_n_ablation_inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dependecies

In [ ]:
import scipy.io as io
import os
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
from google.colab import drive
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve
import h5py
from contextlib import ExitStack
from torch.utils import data
import matplotlib.gridspec as gridspec
import joblib
import seaborn as sns
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
from torch import signal

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


## Install Torch Geometric

In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.9 MB/s eta 0:00:00


## Common functions and naming convention

### Signal Names and their colors

In [ ]:
signal_names = {0: 'Phase A', 1: 'Phase B', 2: 'Phase C',
                3: r'Velocity (rad/s)', 4: 'Position (rad)'}

color_names = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green', 3: 'tab:red',
               4: 'tab:purple', 5: 'tab:brown', 6:'tab:black'} ## some extra colors

### EMA component name map

In [ ]:
component_name = {
    'motor': 'Component: Motor',
    'inverter': 'Component: Inverter'
}

### Model architecture type map

In [ ]:
model_name = {
    'fm': 'Full Model',
    'am': 'Ablation Model'
}

### Color Map

In [ ]:
color_map = {
    'fm': 'mako',
    'am': 'rocket'
}

### Fault type Map (test data type)

In [ ]:
fault_type = {
    'L0' : 'Inverter Normal Dataset',
    'L10' : 'Inveter Faulty with L=0.10',
    'L20' : 'Inverter Faulty with L=0.20',
    'L50' : 'Faulty with L=0.50',
    'L_all' : 'All Inverter Datasets',
    'motor' : 'Motor Test Data',
    'all' : 'All Test Data'
}

### Signal reconstruction function: Using Overlap-Add (OLA) method

In [ ]:
def signal_reconstruct(decoded, stride, total_length):
    """
    Reconstructs original signals from overlapping windows using NumPy.

    Args:
        decoded: Array of shape (B_window, N, W)
        stride: The step size used during windowing
        total_length: The original length of the signal (e.g., 500)

    Returns:
        final_reconstruction: Array of shape (num_samples, N, total_length)
    """
    B_window, N, W = decoded.shape

    # Copy the array so you don't accidentally modify your original data outside the function
    decoded_mod = decoded.copy()

    # Replace indices 0,1,2,3,4 with indices 5,6,7,8,9
    decoded_mod[:, :, :5] = decoded_mod[:, :, 5:10]
    decoded_mod[:, :, 250:255] = decoded_mod[:, :, 245:250]

    # Calculate number of windows per original sample
    K_per_sample = math.floor((total_length - W) / stride + 1)
    num_samples = B_window // K_per_sample

    # Reshape to separate the original samples: (num_samples, windows_per_sample, N, W)
    decoded_grouped = decoded_mod.reshape(num_samples, K_per_sample, N, W)

    # Initialize accumulation and count arrays
    reconstructed = np.zeros((num_samples, N, total_length))
    count = np.zeros((num_samples, N, total_length))

    for sample_idx in range(num_samples):
        for window_idx in range(K_per_sample):
            # Calculate positions
            start_pos = window_idx * stride
            end_pos = start_pos + W

            # Handle potential edge cases where window might exceed total_length
            actual_end = min(end_pos, total_length)
            slice_w = actual_end - start_pos

            # Accumulate values and counts for averaging overlaps
            reconstructed[sample_idx, :, start_pos:actual_end] += \
                decoded_grouped[sample_idx, window_idx, :, :slice_w]

            count[sample_idx, :, start_pos:actual_end] += 1

    # Average by the overlap count (using np.maximum to avoid division by zero)
    final_reconstruction = reconstructed / np.maximum(count, 1)

    return final_reconstruction

### Destandardization Function

In [ ]:
# De-standardization
def destandardize(signal_norm, scaler, N_norm=4):
    # signal_norm: (S, 6, T) -> [iA, iB, iC, omega, sin(theta_M), cos(theta_M)]
    if not torch.is_tensor(signal_norm):
        signal_norm = torch.from_numpy(np.asarray(signal_norm))
    signal_norm = signal_norm.detach().cpu().float()
    S, _, T = signal_norm.shape

    scaled_part = signal_norm[:, :N_norm, :]     # iA, iB, iC, omega
    untouched_part = signal_norm[:, N_norm:, :]  # sin, cos

    # (S, N, T) -> (S, T, N) -> (S*T, N)
    scaled_flat = scaled_part.permute(0, 2, 1).reshape(-1, N_norm)
    original_flat = scaler.inverse_transform(scaled_flat.numpy())

    # (S*T, N) -> (S, T, N) -> (S, N, T)
    original = torch.tensor(original_flat, dtype=torch.float32) \
                    .view(S, T, N_norm).permute(0, 2, 1)

    return torch.cat((original, untouched_part), dim=1)

### Inspection window preparation function

In [ ]:
def prepare_inspection_windows(arr, t_inspec):
    """
    Trims the last time step of an array of shape (s, n, t) and reshapes it
    into continuous inspection windows of shape (s, n, t // t_inspec, t_inspec).

    Parameters:
    -----------
    arr : numpy.ndarray
        Input array of shape (s, n, t)
    t_inspec : int
        The length of each inspection window (e.g., 100)

    Returns:
    --------
    numpy.ndarray
        Reshaped array of shape (s, n, num_windows, t_inspec)
    """
    if arr.ndim == 3:
        # Step 1: Drop the very last time step along the last axis (axis 2)
        # e.g., changing the time dimension from 20001 to 20000
        arr_trimmed = arr[:, :, :-1]

        s, n, t_trimmed = arr_trimmed.shape
        num_windows = t_trimmed // t_inspec

        # Step 2: Direct reshape
        # Because time is the last axis in a C-contiguous NumPy array,
        # this cleanly splits the timeline sequentially into consecutive windows.
        reshaped_arr = arr_trimmed.reshape(s, n, num_windows, t_inspec)
    else:
        arr_trimmed = arr[:, :-1]

        s, t_trimmed = arr_trimmed.shape
        num_windows = t_trimmed // t_inspec

        reshaped_arr = arr_trimmed.reshape(s, num_windows, t_inspec)

    return reshaped_arr

## Mount the Google Drive folder

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/My\ Drive/MSc_Thesis/c10_m1_t0p5_full_model

Mounted at /content/gdrive
/content/gdrive/My Drive/MSc_Thesis/c10_m1_t0p5_full_model


## See what's inside the folder

In [ ]:
print("Dataset folder contains: \n")
print(os.listdir("/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model")) # changes the folder accordingly

Dataset folder contains: 

['data', 'trained_model_state', 'data_inferencing_results', 'c10_m1_t0p5_full_n_ablation_model.ipynb', 'c10_m1_t0p5_results.ipynb', 'hpc_trained_model_state', 'c10_m1_t0p5_full_n_ablation_inferencing.ipynb']


## Standardize the test data with 'Motor' Train Scaler and prepare dataloaders

### Load the standardizing scaler

In [ ]:
# get into data folder
%cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data

# load the scaler
raw_to_norm_scaler = joblib.load("scaler_for_inferencing.joblib")

# get back to main folder
!cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model

/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data


### Load the test data (All Motor and all Inverter)

- Order: $i_A$, $i_B$, $i_C$, $\omega$, $\mathrm{sin}(\theta)$, and $\mathrm{cos}(\theta)$ $←$ order verified from the local python script

In [ ]:
# get into data folder
%cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data

motor_train = np.load("motor_train.npy")
# print(motor_train.shape)

motor_val = np.load("motor_val.npy")
# print(motor_test.shape)

motor_test = np.load("motor_test.npy")
# print(motor_test.shape)

inv_L0 = np.load("inverter_normal.npy")
# print(inv_L0.shape)

inv_L10 = np.load("inverter_fault_L10.npy")
# print(inv_L10.shape)

inv_L20 = np.load("inverter_fault_L20.npy")
# print(inv_L20.shape)

inv_L50 = np.load("inverter_fault_L50.npy")
# print(inv_L50.shape)

/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data


### Dataset class for 'normalization' and 'windowing'

In [ ]:
class EMA_Motor_Dataset(Dataset):
    def __init__(self, raw_data, scaler, window_size=256, stride=128):
       # iA, iB, iC, omega, sin(theta_M), cos(theta_M), time (samples, N, T)
      signals_to_normalize = raw_data[:, [0,1,2,3], :]
      signals_last = raw_data[:, [4,5], :]

      # Convert signals to torch.tensor before permuting
      signals_to_normalize = torch.tensor(signals_to_normalize, dtype=torch.float32)
      signals_last = torch.tensor(signals_last, dtype=torch.float32)

      S, N_norm, T = signals_to_normalize.shape # (S: samples, N: signals, T: time-steps)

      # (S*T, N) > (S, N, T)
      signals_to_normalize_flat = signals_to_normalize.permute(0, 2, 1).reshape(-1, N_norm)
      # Fix: Use signals_flat and convert to numpy for scaler.transform
      signals_to_normalize_scaled = scaler.transform(signals_to_normalize_flat.numpy())
      signals_norm = torch.tensor(signals_to_normalize_scaled, dtype=torch.float32).view(S, T, N_norm).permute(0, 2, 1)
      signal_norm = torch.cat((signals_norm, signals_last), dim=1)
      # create windows
      self.variables_good = self.create_windows(signal_norm, window_size, stride)

    # window creation function
    def create_windows(self, signals, window_size, stride):
        """
        signals: Tensor of shape (samples, num_signals, num_timesteps)          (S, N, T)
        returns: Tensor of shape (num_windows, num_signals, window_length)      (B, N, W)
        """
        S, N, T = signals.shape
        windows = []

        for i in range(S):                                                      # loop over samples
          for start in range(0, T - window_size+1, stride):                     # stride over time
              window = signals[i, :, start:start + window_size]
              windows.append(window)
        return torch.stack(windows)

    def __len__(self):
        return len(self.variables_good)

    def __getitem__(self, idx):
        return self.variables_good[idx]

### Dataset instances for the data

In [ ]:
# define window properties
L = 256       # length
ss = 128       # stride length
T_motor_train = 50001 # time steps (50s, 1kHz)
T_motor_test = 5001 # time steps (5s, 1kHz)
T_inv = 20001      # time steps (20s, 1kHz)

num_windows_motor_train = math.floor((T_motor_train - L) // ss + 1) # same for validation
num_windows_motor_test = math.floor((T_motor_test - L) // ss + 1)
num_windows_inv_test = math.floor((T_inv - L) // ss + 1)

# create dataset

# Motor
motor_train_dataset = EMA_Motor_Dataset(motor_train, raw_to_norm_scaler, window_size=L, stride=ss)
motor_val_dataset = EMA_Motor_Dataset(motor_val, raw_to_norm_scaler, window_size=L, stride=ss)
motor_test_dataset = EMA_Motor_Dataset(motor_test, raw_to_norm_scaler, window_size=L, stride=ss)

# Inverter
inv_L0_dataset = EMA_Motor_Dataset(inv_L0, raw_to_norm_scaler, window_size=L, stride=ss)
inv_L10_dataset = EMA_Motor_Dataset(inv_L10, raw_to_norm_scaler, window_size=L, stride=ss)
inv_L20_dataset = EMA_Motor_Dataset(inv_L20, raw_to_norm_scaler, window_size=L, stride=ss)
inv_L50_dataset = EMA_Motor_Dataset(inv_L50, raw_to_norm_scaler, window_size=L, stride=ss)

print(f"total motor train and test dataset windows from {int(len(motor_train_dataset)/num_windows_motor_train)} samples: {len(motor_train_dataset)}")
print(f"total motor test dataset windows from {int(len(motor_test_dataset)/num_windows_motor_test)} samples: {len(motor_test_dataset)}")
print(f"total inverter 'any kind' test dataset windows from {int(len(inv_L0_dataset)/num_windows_inv_test)} samples: {len(inv_L0_dataset)}")

total motor train and test dataset windows from 48 samples: 18672
total motor test dataset windows from 80 samples: 3040
total inverter 'any kind' test dataset windows from 30 samples: 4650


### Dataloader

In [ ]:
# Motor
motor_train_loader = DataLoader(motor_train_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(motor_train_loader))

motor_val_loader = DataLoader(motor_val_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(motor_val_loader))

motor_test_loader = DataLoader(motor_test_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(motor_test_loader))



# Inverter
inv_L0_loader = DataLoader(inv_L0_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(inv_L0_loader))

inv_L10_loader = DataLoader(inv_L10_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(inv_L10_loader))

inv_L20_loader = DataLoader(inv_L20_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(inv_L20_loader))

inv_L50_loader = DataLoader(inv_L50_dataset, batch_size=32, shuffle=False)
print("Number of batches made using train_loader:", len(inv_L50_loader))

Number of batches made using train_loader: 584
Number of batches made using train_loader: 159
Number of batches made using train_loader: 95
Number of batches made using train_loader: 146
Number of batches made using train_loader: 146
Number of batches made using train_loader: 146
Number of batches made using train_loader: 146


## Model architecture blocks

### Encoder Class

In [ ]:
class LSTM_Encoder(nn.Module):
    def __init__(self, hidden_size=64, bidirectional=True):
        super().__init__()
        self.hidden_size = hidden_size
        D = 2 if bidirectional else 1

        # Note: LSTM features that are not mentioned are taken to be default
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.LeakyReLU(negative_slope=0.01),
            nn.MaxPool1d(kernel_size=2), # 256 -> 128

            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1), # 128 > 64
            nn.LeakyReLU(negative_slope=0.01),
            nn.MaxPool1d(kernel_size=2), # 64 -> 32
            )

        self.bilstm = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=bidirectional
        )

        # Fix: Update flat_size to reflect actual sequence length from CNN/MaxPool layers
        self.flat_size = 32 * (D * hidden_size)
        self.proj = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(self.flat_size, 256),  # 4096 -> 256
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(256),
            nn.Dropout(0.1),
            nn.Linear(256, 128),    # 256 -> 128
            nn.LeakyReLU(negative_slope=0.01),
            nn.LayerNorm(128),
        )

    def forward(self, x):
        # (h0, c0) are tensors of zeros (default)
        B, N, W = x.shape                                                       #(batch_size, num_signals, window_size)

        # Flatten B and N to process all signal windows in parallel
        x = x.view(B * N, 1, W)                                                 # (B*N, 1, W)
        x = self.cnn(x)                                                         # (B*N, 32, W/8=32)
        x = x.permute(0, 2, 1)                                                  # (B*N, 32, 32)
        x, _ = self.bilstm(x)                                                   # (B*N, 32, 128) [ignoring h_n, c_n]
        # x = x.mean(dim=1)                                                       # (B*N, 2H) [Temporal Pooling]


        # Project to target hidden size H
        x = self.proj(x)                                                        # (B*N, 128) [sequential]
        x = x.view(B, N, -1)                                                    # (B, N, 128) [shape restoration]

        return x

### GAT Block

In [ ]:
from torch_geometric.nn import GATConv

In [ ]:
class GAT_Layer(nn.Module):
    def __init__(self, gat_hidden_channels=128, gat_heads=4):
        super().__init__()

        # Layer 1: Takes LSTM features -> GAT Hidden
        self.gat1 = GATConv(in_channels=gat_hidden_channels,
                            out_channels=gat_hidden_channels,
                            heads=gat_heads,
                            concat=True,
                            )

        # Layer 2:
        # We set concat=False to average the heads for the final representation
        self.gat2 = GATConv(in_channels=gat_hidden_channels * gat_heads,
                            out_channels=gat_hidden_channels,
                            heads=1,
                            concat=False,
                            )

        # self.alpha = nn.Parameter(torch.tensor(1.0))        # 100% of the GAT output
        # self.beta = nn.Parameter(torch.tensor(0.4))         # 40% of the LSTM output
        self.gate_weights = nn.Parameter(torch.tensor([0.0, 0.0]))
        # self.out_linear = nn.Linear(2*gat_hidden_channels, gat_hidden_channels) # to cover up for the concatenation at the end


    def forward(self, x):

      B, N, G = x.shape  # x shape from LSTM: [batch_size, num_signals, features=G]; Here G = 128
      device = x.device

      # finding edge indices
      adj = torch.ones((N, N), device=device)                                   # (6, 6)
      idx = adj.nonzero().t()                                                   # (2, 36)
      offsets = torch.arange(B, device=device) * N
      offsets = offsets.view(1, -1, 1)
      full_edge_index = idx.unsqueeze(1) + offsets
      full_edge_index = full_edge_index.transpose(1, 2).reshape(2, -1)

      # GAT Message Passing
      x_flat = x.view(B * N, -1)                                                # (B*N, 128)

      # Layer 1
      h = self.gat1(x_flat, full_edge_index)                                    # (B*N, 128*heads)
      h = F.elu(h)                                                              # (B*N, 128*heads)

      # Layer 2
      h = self.gat2(h, full_edge_index)                                         # (B*N, 128)

      # 4. Residual Connection (Crucial to prevent "Same Result" collapse)                                               # (B, N, 128) [elementwise addition]

      # # trainable alpha and beta logic without lineat and concat
      weights = torch.softmax(self.gate_weights, dim=0)
      alpha = weights[0]
      beta = weights[1]
      out = alpha * h.view(B, N, -1) + beta * x

      # # Linear layer + concatenation logic
      # out = torch.cat((self.alpha * h.view(B, N, -1), self.beta * x), dim=-1)
      # out = self.out_linear(out)

      # # without skip connection
      # out = h.view(B, N, -1)

      return out

### Decoder

In [ ]:
class LSTM_Decoder(nn.Module):
    def __init__(self, hidden_size=128, window_size=256):
        super().__init__()
        self.window_size = window_size
        self.hidden_size = hidden_size
        self.gat_output_dim = 128

        # hidden state initialization
        self.project = nn.Linear(self.gat_output_dim, hidden_size)

        # The LSTM now takes the GAT feature + a 1D value as input at each step
        # This is more robust than zeros alone.
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.05)

        self.out_linear = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.LeakyReLU(negative_slope=0.01),

            nn.Linear(64, 16),
            nn.LeakyReLU(negative_slope=0.01),

            #nn.Dropout(0.02),
            nn.Linear(16, 1))

        #self.amplitude_gain = nn.Parameter(torch.tensor(2.0))

    def forward(self, x):
        # x: (B, N, G) -> (batch_size, num_signals, gat_output)
        B, N, G = x.shape

        x_flat = x.view(B * N, G)                                               # (B*N, G)

        # # tanh initialization (free magnitudes, less stable)
        # h0_initial = torch.tanh(self.project(x_flat))                                       # (B*N, H_dec)
        # h0 = h0_initial.unsqueeze(0).repeat(self.lstm.num_layers, 1, 1)         # (num_layers,
        # c0 = h0_initial.unsqueeze(0).repeat(self.lstm.num_layers, 1, 1)         # B*N, H_dec)

        # Linear initialization
        h0 = self.project(x_flat).unsqueeze(0).repeat(self.lstm.num_layers, 1, 1)
        c0 = self.project(x_flat).unsqueeze(0).repeat(self.lstm.num_layers, 1, 1)

        # lets create a time ramp
        time_ramp = torch.linspace(0, 1, self.window_size).to(x.device)
        time_ramp = time_ramp.view(1, self.window_size, 1).repeat(B*N, 1, 1)

        # # We repeat the GAT vector 200 times. This "feeds" the context
        # # to the LSTM at every single time step.
        # context = x_flat.unsqueeze(1).repeat(1, self.window_size, 1)            # (B*N, W, G)

        # decoder_input = torch.cat((context, time_ramp), dim=-1)                 # (B*N, W, G+1)

        #out, _ = self.lstm(decoder_input, (h0, c0))                             # (B*N, W, H_dec)
        out, _ = self.lstm(time_ramp, (h0, c0))

        out = self.out_linear(out)                                              # (B*N, W, 1)
        return out.view(B, N, self.window_size)           # (B, N, W)

## Full Model

In [ ]:
class CombinedFullModel(nn.Module):
    def __init__(self, enc_hidden=64, gat_hidden=128, gat_heads=4, dec_hidden=128, window_size=256):
    #def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = LSTM_Encoder(hidden_size=enc_hidden, bidirectional=True)
        self.gat = GAT_Layer(gat_hidden_channels=gat_hidden, gat_heads=gat_heads)
        self.decoder = LSTM_Decoder(hidden_size=dec_hidden, window_size=window_size)

    def forward(self, x):
        # x: (B, N, W)

        enc = self.encoder(x)   # → (B, N, 2H)
        gat_out = self.gat(enc) # → (B, N, G=2H)
        dec = self.decoder(gat_out)  # → (B, N, W)

        return dec

## Ablation Model

In [ ]:
class CombinedAblationModel(nn.Module):
    def __init__(self, enc_hidden=64, dec_hidden=128, window_size=256):
        super().__init__()
        self.encoder = LSTM_Encoder(hidden_size=enc_hidden, bidirectional=True)
        #self.gat = GAT_Layer(gat_hidden_channels=gat_hidden, gat_heads=gat_heads)
        self.decoder = LSTM_Decoder(hidden_size=dec_hidden, window_size=window_size)

    def forward(self, x):
        # x: (B, N, W)

        enc = self.encoder(x)   # → (B, N, 2H)
        #gat_out = self.gat(enc) # → (B, N, G=2H)
        dec = self.decoder(enc)  # → (B, N, W)

        return dec

## Instantiate both model (full and ablation)

In [ ]:
full_model = CombinedFullModel()
full_model.to(device)

CombinedFullModel(
  (encoder): LSTM_Encoder(
    (cnn): Sequential(
      (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): LeakyReLU(negative_slope=0.01)
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,))
      (4): LeakyReLU(negative_slope=0.01)
      (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
    (proj): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=256, bias=True)
      (2): LeakyReLU(negative_slope=0.01)
      (3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): Dropout(p=0.1, inplace=False)
      (5): Linear(in_features=256, out_features=128, bias=True)
      (6): LeakyReLU(negative_slope=0.01)
      (7): LayerNorm((128,), eps=1e-05, elementwi

In [ ]:
ablation_model = CombinedAblationModel()
ablation_model.to(device)

CombinedAblationModel(
  (encoder): LSTM_Encoder(
    (cnn): Sequential(
      (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): LeakyReLU(negative_slope=0.01)
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,))
      (4): LeakyReLU(negative_slope=0.01)
      (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
    (proj): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=256, bias=True)
      (2): LeakyReLU(negative_slope=0.01)
      (3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): Dropout(p=0.1, inplace=False)
      (5): Linear(in_features=256, out_features=128, bias=True)
      (6): LeakyReLU(negative_slope=0.01)
      (7): LayerNorm((128,), eps=1e-05, eleme

## Loading model states (setting up for inferencing)

#### Full Model

In [ ]:
# get into the state folder
%cd /content/gdrive/My\Drive/MSc_Thesis/c10_m1_t0p5_full_model/hpc_trained_model_state

# # Load the saved state dict
# full_checkpoint = torch.load('c10_m1_t0p5_Full_Epochs:500_time:17-09-2026_18-34-16.pth', map_location=torch.device('cpu'))
# full_model.load_state_dict(full_checkpoint['fm_model_state_dict'])

# Load the HPC trained state dict
full_checkpoint_fm = torch.load('hpc_c10_m1_t0p5_model_final_fm.pth', map_location=torch.device('cpu'), weights_only=False)
state_dict_fm = {k: v for k, v in full_checkpoint_fm['model_state_dict'].items() if k != 'decoder.time_ramp'}
full_model.load_state_dict(state_dict_fm)

# Set to evaluation mode — critical for inference
full_model.eval()

/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/hpc_trained_model_state


CombinedFullModel(
  (encoder): LSTM_Encoder(
    (cnn): Sequential(
      (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): LeakyReLU(negative_slope=0.01)
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,))
      (4): LeakyReLU(negative_slope=0.01)
      (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
    (proj): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=256, bias=True)
      (2): LeakyReLU(negative_slope=0.01)
      (3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): Dropout(p=0.1, inplace=False)
      (5): Linear(in_features=256, out_features=128, bias=True)
      (6): LeakyReLU(negative_slope=0.01)
      (7): LayerNorm((128,), eps=1e-05, elementwi

#### Ablation Model

In [ ]:
# # Load the saved state dict
# ablation_checkpoint = torch.load('c10_m1_t0p5_Ablation_Epochs:500_time:17-09-2026_18-34-16.pth', map_location=torch.device('cpu'))
# ablation_model.load_state_dict(ablation_checkpoint['am_model_state_dict'])

# Load the HPC trained state dict
full_checkpoint_am = torch.load('hpc_c10_m1_t0p5_model_final_am.pth', map_location=torch.device('cpu'), weights_only=False)
state_dict_am = {k: v for k, v in full_checkpoint_am['model_state_dict'].items() if k != 'decoder.time_ramp'}
ablation_model.load_state_dict(state_dict_am)

# Set to evaluation mode — critical for inference
ablation_model.eval()

CombinedAblationModel(
  (encoder): LSTM_Encoder(
    (cnn): Sequential(
      (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): LeakyReLU(negative_slope=0.01)
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,))
      (4): LeakyReLU(negative_slope=0.01)
      (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
    (proj): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=256, bias=True)
      (2): LeakyReLU(negative_slope=0.01)
      (3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): Dropout(p=0.1, inplace=False)
      (5): Linear(in_features=256, out_features=128, bias=True)
      (6): LeakyReLU(negative_slope=0.01)
      (7): LayerNorm((128,), eps=1e-05, eleme

## Inferencing function

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()  # Switch to evaluation mode
    total_loss = 0.0
    total_accuracy = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_windows in loader:
            # Prepare data (same as training)
            batch_windows = batch_windows.to(device).float()

            # Forward pass
            model_output = model(batch_windows)

            # Store for visualization/metrics
            all_predictions.append(model_output.cpu().numpy())
            all_targets.append(batch_windows.cpu().numpy())

    return np.concatenate(all_predictions), np.concatenate(all_targets)

## Data (256 long windows) generation from trained models (Inferencing)

In [ ]:
# -------------------------------------------Full Model---------------------------------------------------------------

# Motor
motor_train_fm_predictions, motor_train_fm_targets = evaluate_model(full_model, motor_train_loader, device)
motor_val_fm_predictions, motor_val_fm_targets = evaluate_model(full_model, motor_val_loader, device)
motor_test_fm_predictions, motor_test_fm_targets = evaluate_model(full_model, motor_test_loader, device)


# Inverter
inv_L0_fm_predictions, inv_L0_fm_targets = evaluate_model(full_model, inv_L0_loader, device)
inv_L10_fm_predictions, inv_L10_fm_targets = evaluate_model(full_model, inv_L10_loader, device)
inv_L20_fm_predictions, inv_L20_fm_targets = evaluate_model(full_model, inv_L20_loader, device)
inv_L50_fm_predictions, inv_L50_fm_targets = evaluate_model(full_model, inv_L50_loader, device)

#-------------------------------------------Ablation Model------------------------------------------------------------

# Motor
motor_train_am_predictions, motor_train_am_targets = evaluate_model(ablation_model, motor_train_loader, device)
motor_val_am_predictions, motor_val_am_targets = evaluate_model(ablation_model, motor_val_loader, device)
motor_test_am_predictions, motor_test_am_targets = evaluate_model(ablation_model, motor_test_loader, device)


# Inverter
inv_L0_am_predictions, inv_L0_am_targets = evaluate_model(ablation_model, inv_L0_loader, device)
inv_L10_am_predictions, inv_L10_am_targets = evaluate_model(ablation_model, inv_L10_loader, device)
inv_L20_am_predictions, inv_L20_am_targets = evaluate_model(ablation_model, inv_L20_loader, device)
inv_L50_am_predictions, inv_L50_am_targets = evaluate_model(ablation_model, inv_L50_loader, device)

## Saving 256 long windows (actual and reconstructed)


In [ ]:
#--------------------------------------------Full Model---------------------------------------------------------------

# Create a master dictionary for the full model results
fm_inferencing_results_windows = {
    'motor_train_fm_predictions': motor_train_fm_predictions, 'motor_train_fm_targets': motor_train_fm_targets,
    'motor_val_fm_predictions': motor_val_fm_predictions, 'motor_val_fm_targets': motor_val_fm_targets,
    'motor_test_fm_predictions': motor_test_fm_predictions, 'motor_test_fm_targets': motor_test_fm_targets,

    'inv_L0_fm_predictions': inv_L0_fm_predictions, 'inv_L0_fm_targets': inv_L0_fm_targets,
    'inv_L10_fm_predictions': inv_L10_fm_predictions, 'inv_L10_fm_targets': inv_L10_fm_targets,
    'inv_L20_fm_predictions': inv_L20_fm_predictions, 'inv_L20_fm_targets': inv_L20_fm_targets,
    'inv_L50_fm_predictions': inv_L50_fm_predictions, 'inv_L50_fm_targets': inv_L50_fm_targets,
}

#-------------------------------------------Ablation Model------------------------------------------------------------
am_inferencing_results_windows = {
    'motor_train_am_predictions': motor_train_am_predictions, 'motor_train_am_targets': motor_val_am_targets,
    'motor_val_am_predictions': motor_val_am_predictions, 'motor_val_am_targets': motor_val_am_targets,
    'motor_test_am_predictions': motor_test_am_predictions, 'motor_test_am_targets': motor_test_am_targets,

    'inv_L0_am_predictions': inv_L0_am_predictions, 'inv_L0_am_targets': inv_L0_am_targets,
    'inv_L10_am_predictions': inv_L10_am_predictions, 'inv_L10_am_targets': inv_L10_am_targets,
    'inv_L20_am_predictions': inv_L20_am_predictions, 'inv_L20_am_targets': inv_L20_am_targets,
    'inv_L50_am_predictions': inv_L50_am_predictions, 'inv_L50_am_targets': inv_L50_am_targets,
}
# get into data folder
%cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data_inferencing_results/hpc_trained

# Save as two clean compressed archive files
np.savez_compressed('full_model_inferencing_results_256_windows.npz', **fm_inferencing_results_windows)
np.savez_compressed('ablation_model_inferencing_results_256_windows.npz', **am_inferencing_results_windows)

# get back to main folder
!cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model

print("Reconstructed windows successfully saved to compressed archives!")

/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data_inferencing_results/hpc_trained
Reconstructed windows successfully saved to compressed archives!


## Reconstructing the signals with OLA and then destandardizing

In [ ]:
import numpy as np
import torch

#--------------------------------------------Full Model---------------------------------------------------------------

# Motor
# motor_train_fm_recon_signal = signal_reconstruct(torch.from_numpy(motor_train_fm_predictions).to(device), stride=ss, total_length=50001) # reconstruction
# motor_train_fm_recon_signal = destandardize(motor_train_fm_recon_signal, raw_to_norm_scaler)                                            # reverse normalization

motor_train_fm_recon_signal = signal_reconstruct(motor_train_fm_predictions, stride=ss, total_length=50001) # reconstruction
motor_train_fm_recon_signal = destandardize(motor_train_fm_recon_signal, raw_to_norm_scaler)

motor_val_fm_recon_signal = signal_reconstruct(motor_val_fm_predictions, stride=ss, total_length=50001) # reconstruction
motor_val_fm_recon_signal = destandardize(motor_val_fm_recon_signal, raw_to_norm_scaler)                                            # reverse normalization

motor_test_fm_recon_signal = signal_reconstruct(motor_test_fm_predictions, stride=ss, total_length=5001) # reconstruction
motor_test_fm_recon_signal = destandardize(motor_test_fm_recon_signal, raw_to_norm_scaler)                                            # reverse normalization

# Invereter
inv_L0_fm_recon_signal = signal_reconstruct(inv_L0_fm_predictions, stride=ss, total_length=20001)
inv_L0_fm_recon_signal = destandardize(inv_L0_fm_recon_signal, raw_to_norm_scaler)

inv_L10_fm_recon_signal = signal_reconstruct(inv_L10_fm_predictions, stride=ss, total_length=20001)
inv_L10_fm_recon_signal = destandardize(inv_L10_fm_recon_signal, raw_to_norm_scaler)

inv_L20_fm_recon_signal = signal_reconstruct(inv_L20_fm_predictions, stride=ss, total_length=20001)
inv_L20_fm_recon_signal = destandardize(inv_L20_fm_recon_signal, raw_to_norm_scaler)

inv_L50_fm_recon_signal = signal_reconstruct(inv_L50_fm_predictions, stride=ss, total_length=20001)
inv_L50_fm_recon_signal = destandardize(inv_L50_fm_recon_signal, raw_to_norm_scaler)

#-------------------------------------------Ablation Model------------------------------------------------------------

# Motor
motor_train_am_recon_signal = signal_reconstruct(motor_train_am_predictions, stride=ss, total_length=50001)
motor_train_am_recon_signal = destandardize(motor_train_am_recon_signal, raw_to_norm_scaler)

motor_val_am_recon_signal = signal_reconstruct(motor_val_am_predictions, stride=ss, total_length=50001)
motor_val_am_recon_signal = destandardize(motor_val_am_recon_signal, raw_to_norm_scaler)

motor_test_am_recon_signal = signal_reconstruct(motor_test_am_predictions, stride=ss, total_length=5001)
motor_test_am_recon_signal = destandardize(motor_test_am_recon_signal, raw_to_norm_scaler)

# Inverter
inv_L0_am_recon_signal = signal_reconstruct(inv_L0_am_predictions, stride=ss, total_length=20001)
inv_L0_am_recon_signal = destandardize(inv_L0_am_recon_signal, raw_to_norm_scaler)

inv_L10_am_recon_signal = signal_reconstruct(inv_L10_am_predictions, stride=ss, total_length=20001)
inv_L10_am_recon_signal = destandardize(inv_L10_am_recon_signal, raw_to_norm_scaler)

inv_L20_am_recon_signal = signal_reconstruct(inv_L20_am_predictions, stride=ss, total_length=20001)
inv_L20_am_recon_signal = destandardize(inv_L20_am_recon_signal, raw_to_norm_scaler)

inv_L50_am_recon_signal = signal_reconstruct(inv_L50_am_predictions, stride=ss, total_length=20001)
inv_L50_am_recon_signal = destandardize(inv_L50_am_recon_signal, raw_to_norm_scaler)

## Saving fully reconstructed signals

NOTE: Actual signals are not reconstructed from the model targets. Refer raw data for comparison.

In [ ]:
#--------------------------------------------Full Model---------------------------------------------------------------

# Create a master dictionary for the full model results
fm_inferencing_results_recon_signals = {
    'motor_train_fm_recon_signal': motor_train_fm_recon_signal,
    'motor_val_fm_recon_signal': motor_val_fm_recon_signal,
    'motor_test_fm_recon_signal': motor_test_fm_recon_signal,

    'inv_L0_fm_recon_signal': inv_L0_fm_recon_signal,
    'inv_L10_fm_recon_signal': inv_L10_fm_recon_signal,
    'inv_L20_fm_recon_signal': inv_L20_fm_recon_signal,
    'inv_L50_fm_recon_signal': inv_L50_fm_recon_signal,
}

#-------------------------------------------Ablation Model------------------------------------------------------------
am_inferencing_results_recon_signals = {
    'motor_train_am_recon_signal': motor_train_am_recon_signal,
    'motor_val_am_recon_signal': motor_val_am_recon_signal,
    'motor_test_am_recon_signal': motor_test_am_recon_signal,

    'inv_L0_am_recon_signal': inv_L0_am_recon_signal,
    'inv_L10_am_recon_signal': inv_L10_am_recon_signal,
    'inv_L20_am_recon_signal': inv_L20_am_recon_signal,
    'inv_L50_am_recon_signal': inv_L50_am_recon_signal,
}
# get into data folder
%cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data_inferencing_results/hpc_trained

# Save as two clean compressed archive files
np.savez_compressed('full_model_inferencing_results_reconstructed_signals.npz', **fm_inferencing_results_recon_signals)
np.savez_compressed('ablation_model_inferencing_results_reconstructed_signals.npz', **am_inferencing_results_recon_signals)

# get back to main folder
!cd /content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model

print("Fully reconstructed signals successfully saved to compressed archives!")

/content/gdrive/MyDrive/MSc_Thesis/c10_m1_t0p5_full_model/data_inferencing_results/hpc_trained
Fully reconstructed signals successfully saved to compressed archives!
